In [ ]:
#################################
#IMPORTS
#################################

import os
import shutil
import csv
import torch
import cv2
import numpy as np
import tensorflow as tf
import unicodedata
import easyocr
import pandas as pd
import Levenshtein
import ast
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
from PIL import Image, ImageChops, ImageEnhance
from models.experimental import attempt_load
from utils.general import non_max_suppression, scale_coords
from utils.datasets import letterbox
from utils.plots import plot_one_box

#################################
#LECTURE SANS ACCENT
#################################

# Sert à lire les images dont les noms ont un accent
def remove_accents(input_str):
    nfkd_form = unicodedata.normalize('NFKD', input_str)
    return "".join([c for c in nfkd_form if not unicodedata.combining(c)])

#################################
#DETECTION & SPOOL 
#################################

# Sert à charger les 2 modèles YOLO et SPOOL
def load_models():
    # Chemin du modèle yolo
    weights_path = 'best.pt'
    # Cuda si gpu Nvidia sinon Processeur CPU 
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_yolo = attempt_load(weights_path, map_location=device)
    model_yolo.eval()
    # Chemin du modèle de classification spool
    model_spool = tf.saved_model.load('spool_profond Tensorflow')
    # Chemin du fichier de labels pour les classes de spool
    spool_labels_path = os.path.join('spool_profond Tensorflow', 'labels.txt')
    with open(spool_labels_path, 'r') as f:
        spool_class_names = f.read().splitlines()
    return model_yolo, model_spool, spool_class_names, device

# Fonction DETECTION
def detect_objects(model_yolo, device, img_path):
    img0 = cv2.imread(img_path)
    if img0 is None:
        print(f"Erreur : l'image '{img_path}' n'a pas pu être lue.")
        return [], None, None, None
    
    original_img = img0.copy()
    
    img = letterbox(img0, new_shape=640)[0]
    img = img[:, :, ::-1].transpose(2, 0, 1)
    img = np.ascontiguousarray(img)
    
    img = torch.from_numpy(img).to(device)
    img = img.float()
    img /= 255.0
    if img.ndimension() == 3:
        img = img.unsqueeze(0)

    with torch.no_grad():
        pred = model_yolo(img, augment=False)[0]
    
    pred = non_max_suppression(pred, 0.60, 0.45, agnostic=False)
    
    detected_objects = []
    low_conf_objects = []
    for det in pred:
        if len(det):
            det[:, :4] = scale_coords(img.shape[2:], det[:, :4], img0.shape).round()
            for *xyxy, conf, cls in reversed(det):
                if conf < 0.60:
                    low_conf_objects.append(model_yolo.names[int(cls)])
                else:
                    detected_objects.append(model_yolo.names[int(cls)])
                label = f'{model_yolo.names[int(cls)]} {conf:.2f}'
                plot_one_box(xyxy, img0, label=label, color=(255, 0, 0), line_thickness=2)
    
    return detected_objects, low_conf_objects, img0, original_img

# Création du CSV de l'intervention, avec le num inter, le nombre d'images, la liste des images 
def create_csv(output_csv, interventions_data):
    with open(output_csv, 'w', newline='') as csvfile:
        fieldnames = ['Intervention Number', 'Number of Images', 'Image List']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames, delimiter=',')
        writer.writeheader()
        for intervention in interventions_data:
            writer.writerow({
                'Intervention Number': intervention['number'],
                'Number of Images': intervention['num_images'],
                'Image List': '; '.join(intervention['images'])
            })

# Fonction Classification SPOOL
def classify_spool(model_spool, spool_class_names, image):
    image = cv2.resize(image, (224, 224))
    image = np.expand_dims(image, axis=0)
    image = image.astype(np.float32)
    image = image / 255.0
    infer = model_spool.signatures["serving_default"]
    predictions = infer(tf.constant(image))
    predicted_class = np.argmax(predictions['Confidences'].numpy(), axis=1)[0]
    return spool_class_names[predicted_class]

# Fonction de création des dossiers d'objets détectés 
def create_result_dirs(base_dir, object_names, spool_class_names):
    os.makedirs(base_dir, exist_ok=True)
    for obj_name in object_names:
        os.makedirs(os.path.join(base_dir, obj_name), exist_ok=True)
    for class_name in spool_class_names:
        os.makedirs(os.path.join(base_dir, 'spool', class_name), exist_ok=True)

################################################
# PROCESS DES IMAGES POUR DETECTION ET SPOOL
################################################

def process_images(input_dir, output_dir, model_yolo, model_spool, spool_class_names, device):
    interventions_data = []
    
    object_names = model_yolo.names
    
    create_result_dirs(os.path.join(output_dir, 'objets_detectes'), object_names, spool_class_names)
    
    special_categories = ['pbo_ouvert', 'pbo_ferme', 'chambre_ouvert', 'chambre_ferme']
    banned_keywords = ['signature', 'decharge']
    
    for root, dirs, files in os.walk(input_dir):
        for dir_name in dirs:
            intervention_number = dir_name
            intervention_path = os.path.join(root, dir_name)
            images = [f for f in os.listdir(intervention_path) if f.endswith(('.png', '.jpg', '.jpeg'))]
            
            intervention_info = {
                'number': intervention_number,
                'num_images': len(images),
                'images': []
            }
            
            for img_name in images:
                img_path = os.path.join(intervention_path, img_name)
                
                if any(keyword in img_name.lower() for keyword in banned_keywords):
                    hors_nom_path = os.path.join(output_dir, 'hors_nom', img_name)
                    os.makedirs(os.path.dirname(hors_nom_path), exist_ok=True)
                    shutil.copy(img_path, hors_nom_path)
                    continue
                
                new_img_name = remove_accents(img_name)
                new_img_path = os.path.join(intervention_path, new_img_name)
                os.rename(img_path, new_img_path)
                img_path = new_img_path
                
                detected_objects, low_conf_objects, annotated_img, original_img = detect_objects(model_yolo, device, img_path)
                
                if annotated_img is None:
                    continue
                
                intervention_info['images'].append(new_img_name)
                
                if detected_objects:
                    detected_folder = detected_objects[0]
                    output_path = os.path.join(output_dir, 'objets_detectes', detected_folder, new_img_name)
                    if detected_folder in special_categories:
                        spool_path = os.path.join(output_dir, 'spool', new_img_name)
                        os.makedirs(os.path.dirname(spool_path), exist_ok=True)
                        cv2.imwrite(spool_path, annotated_img)
                        
                        spool_class = classify_spool(model_spool, spool_class_names, annotated_img)
                        spool_class_path = os.path.join(output_dir, 'spool', spool_class, new_img_name)
                        os.makedirs(os.path.dirname(spool_class_path), exist_ok=True)
                        cv2.imwrite(spool_class_path, annotated_img)
                elif low_conf_objects:
                    output_path = os.path.join(output_dir, 'non_sures', new_img_name)
                else:
                    output_path = os.path.join(output_dir, 'sans_objets', new_img_name)
                
                os.makedirs(os.path.dirname(output_path), exist_ok=True)
                cv2.imwrite(output_path, annotated_img)

                #enlever les annotations
                if 'pto' in detected_objects:
                    pto_path = os.path.join(output_dir, 'PTO', new_img_name)
                    os.makedirs(os.path.dirname(pto_path), exist_ok=True)
                    cv2.imwrite(pto_path, original_img)
                
            interventions_data.append(intervention_info)
    
    create_csv(os.path.join(output_dir, 'interventions.csv'), interventions_data)
    #return output_dir

#################################
#NUM PTO ALWU
#################################

#def search_pto(main_folder, destination_folder): 
    # Chemin du dossier principal
    #main_folder = './Download_ARD2'  # Remplacez par le chemin réel de votre dossier ARD2
    # Chemin du dossier de destination
    #destination_folder = './PTO'  # Remplacez par le chemin réel du dossier de destination
 
    # Créer le dossier de destination s'il n'existe pas
    #os.makedirs(destination_folder, exist_ok=True)
 
    # Parcourir tous les sous-dossiers et fichiers dans le dossier principal
    #for root, dirs, files in os.walk(main_folder):
        #for file in files:
            #if 'PTO' in file:
                # Chemin complet du fichier
                #file_path = os.path.join(root, file)
                # Déplacer le fichier vers le dossier de destination
                #shutil.move(file_path, destination_folder)
                # Passer au dossier suivant
                #break

def search_pto(input_folder, destination_folder):
    
    # Créer le dossier de destination s'il n'existe pas
    os.makedirs(destination_folder, exist_ok=True)

    # Parcourir tous les sous-dossiers et fichiers dans le dossier principal
    for root, dirs, files in os.walk(input_folder):
        for file in files:
            if 'PTO' in file:
                # Chemin complet du fichier
                file_path = os.path.join(root, file)
                # Chemin complet du fichier dans le dossier de destination
                destination_path = os.path.join(destination_folder, file)
                
                # Vérifier si le fichier existe déjà dans le dossier de destination
                if not os.path.exists(destination_path):
                    # Déplacer le fichier vers le dossier de destination
                    shutil.move(file_path, destination_path)
                else:
                    print(f"Le fichier {file} existe déjà dans {destination_folder}. Il ne sera pas déplacé.")

    print(f"Tâche de recherche et de déplacement des fichiers PTO terminée.")



#################################
#FAKE 
#################################

# Fonction de détection des photomontages PTO 
def detect_fake_photomontages(output_dir):
    # Chargement du modèle entraîné 
    fake_detection_model = tf.keras.models.load_model('fakedetection_model.h5')
    fake = []

    # Fonction pour générer l'image ELA
    def ela_image(path, quality=60):
        original = Image.open(path).convert('RGB')
        resaved = 'resaved.jpg'
        original.save(resaved, 'JPEG', quality=quality)
        resaved_image = Image.open(resaved)
        ela_image = ImageChops.difference(original, resaved_image)
        extrema = ela_image.getextrema()
        max_diff = max([ex[1] for ex in extrema])
        scale = 255.0 / max_diff
        ela_image = ImageEnhance.Brightness(ela_image).enhance(scale)
        return ela_image

    # Fonction pour prétraiter les images 
    def preprocess_image(img_path, target_size=(128, 128)):
        ela_img = ela_image(img_path)
        ela_img = ela_img.resize(target_size)
        ela_img = np.array(ela_img)
        ela_img = np.expand_dims(ela_img, axis=0)
        ela_img = ela_img / 255.0
        return ela_img

    # Fonction pour faire des prédictions sur de nouvelles images
    def predict_on_new_images(model, img_dir, target_size=(128, 128), threshold=0.48):
        for img_name in os.listdir(img_dir):
            img_path = os.path.join(img_dir, img_name)
            img_array = preprocess_image(img_path, target_size)
            prediction = model.predict(img_array)[0][0]
            predicted_class = "fake" if prediction >= threshold else "real"
            if predicted_class == "fake":
                fake.append(img_name)
            print(f"Prédiction pour {img_name} : {predicted_class} (probabilité : {prediction:.4f})")

    # Création du répertoire de sortie "pto_test"
    new_images_directory = output_dir

    # Appel de la fonction de prédiction
    predict_on_new_images(fake_detection_model, new_images_directory)

    # Création du répertoire des Fakes et pto_test
    fake_output_dir = os.path.join(output_dir, 'fake')
    pto_test_dir = os.path.join(output_dir, 'pto_test')
    os.makedirs(fake_output_dir, exist_ok=True)
    os.makedirs(pto_test_dir, exist_ok=True)

    for img_name in fake:
        shutil.move(os.path.join(new_images_directory, img_name), os.path.join(fake_output_dir, img_name))

    # Déplacer les images réelles dans le dossier pto_test
    for img_name in os.listdir(new_images_directory):
        if img_name not in fake and img_name.endswith(('.png', '.jpg', '.jpeg')):
            shutil.move(os.path.join(new_images_directory, img_name), os.path.join(pto_test_dir, img_name))

    print(fake)
    return fake    

#################################
#NUM PTO 
#################################

# Fonction de rotation de l'image
def rotate_image(image, angle):
    center = (image.shape[1] // 2, image.shape[0] // 2)
    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(image, matrix, (image.shape[1], image.shape[0]))
    return rotated

# Fonction de rotation des bbox de texte 
def rotate_bbox(bbox, angle, image_shape):
    angle = -angle
    center = (image_shape[1] // 2, image_shape[0] // 2)
    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated_bbox = []
    for point in bbox:
        point = np.append(point, 1)
        new_point = np.dot(matrix, point)
        rotated_bbox.append(new_point)
    return np.array(rotated_bbox)

# Fonction pour redimensionner les images en dessous d'une certaine taille
def resize_and_crop_image(image, min_size=(600, 600)):
    height, width = image.shape[:2]
    if height < min_size[0] or width < min_size[1]:
        new_height = max(height, min_size[0])
        new_width = max(width, min_size[1])
        resized_image = cv2.resize(image, (new_width, new_height), interpolation=cv2.INTER_LINEAR)
    else:
        resized_image = image

    # Découper le bas de l'image (environ 1/6 de la hauteur)
    crop_height = int(resized_image.shape[0] * (5/6))
    cropped_image = resized_image[:crop_height, :]
    
    return cropped_image

def color(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Application du flou gaussien pour réduire le bruit 
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Appliquer un seuillage adaptatif
    thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 11, 2)
    return thresh

# Traiter les images dans un dossier 
def process_folder(input_folder, output_folder, results_file, csv_with_numbers, fake):
    
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    
    reader = easyocr.Reader(['fr'], gpu=False)

    # Supprimer le fichier CSV s'il existe déjà
    if os.path.exists(results_file):
        os.remove(results_file)
    
    # Ouvre le fichier CSV en mode ajout, créez-le s'il n'existe pas
    with open(results_file, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['filename', 'text', 'score', 'bbox'])
        
        with open(csv_with_numbers, 'w', newline='') as csv_numbers_file:
            numbers_writer = csv.writer(csv_numbers_file)
            numbers_writer.writerow(['filename', 'number'])

            # Parcourez tous les fichiers du dossier d’entrée
            for filename in os.listdir(input_folder):
                if filename.lower().endswith(('.png', '.jpg', '.jpeg')) and filename not in fake:
                    image_path = os.path.join(input_folder, filename)
                    img_base = cv2.imread(image_path)
                    # Redimensionner l'image si nécessaire
                    img_base = resize_and_crop_image(img_base)
                    img = color(img_base)
                    best_text = None
                    best_score = 0
                    best_bbox = None
                    texts_by_rotation = {}

                    angles = [0, 90, 180, 270]
                    # Variable pour stocker l'angle avec le meilleur texte
                    angle_choix = 0

                    for angle in angles:
                        rotated_img = rotate_image(img, angle)
                        rotated_texts = reader.readtext(rotated_img, allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-#', width_ths=0.7)
                        texts_by_rotation[angle] = rotated_texts
                        for bbox, text, score in rotated_texts:
                            if len(text) > 3:
                                if score > best_score:
                                    best_score = score
                                    best_text = text
                                    best_bbox = rotate_bbox(bbox, angle, img.shape)
                                    angle_choix = angle

                    threshold = 0.2
                    if best_text:
                        for bbox, text, score in texts_by_rotation[angle_choix]:
                            if score > threshold:
                                best_bbox = np.intp(rotate_bbox(bbox, angle_choix, img.shape))
                                cv2.rectangle(img_base, tuple(best_bbox[0]), tuple(best_bbox[2]), (0, 255, 0), 5)
                                cv2.putText(img_base, text, tuple(best_bbox[0]), cv2.FONT_HERSHEY_COMPLEX, 0.65, (255, 0, 0), 2)
                                writer.writerow([filename, text, score, bbox])
                                if text.isdigit():
                                    numbers_writer.writerow([filename, text])
                    else:
                        writer.writerow([filename, 'No text detected', 'N/A', 'N/A'])

                    # Enregistrer l'image annotée dans le dossier de sortie
                    output_image_path = os.path.join(output_folder, filename)
                    cv2.imwrite(output_image_path, img_base)


# Fonction de comparaison des résultats OCR 
def compare_ocr_results(output_dir, fake):
    # Charger les données 
    detected = pd.read_csv(os.path.join(output_dir, 'results_file.csv'))
    expected = pd.read_csv(r"C:\Users\angivord\Music\projet\test_jeudi\outputs\num_PTO.csv", sep=";")

    expected['filename'] = expected['filename'].astype(str)
    detected['filename'] = detected['filename'].str.extract(r'_(\d+)_')[0]
    detected = detected.dropna(subset=['bbox'])
    
    if 'filename' not in detected.columns:
        raise KeyError("'filename' column is missing in detected DataFrame.")
    if 'filename' not in expected.columns:
        raise KeyError("'filename' column is missing in expected DataFrame.")

    # Fonction pour calculer le centre d'une Bbox
    def calculate_bbox_center(bbox):
        x1, y1 = bbox[0]
        x2, y2 = bbox[2]
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        return cx, cy

    # Fonction pour calculer la distance euclidienne entre deux points
    def calculate_distance(point1, point2):
        x1, y1 = point1
        x2, y2 = point2
        distance = np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)
        return distance

    # Fonction pour extraire les coordonnées de BBOX à partir d'une chaîne de caractères
    def extract_bbox_coordinates(bbox_str):
        if pd.isna(bbox_str):
            return None 
        return ast.literal_eval(bbox_str)

    # Seuil de distance pour joindre les textes (adjustable si besoin)
    distance_threshold = 350

    # Liste pour stocker les résultats finaux
    final_results = []

    # Joindre les textes si la distance entre les Bbox est inférieure au seuil
    for filename, group in detected.groupby('filename'):
        if len(group) == 0:
            continue
        
        # Filtrer les lignes avec des scores supérieures à 0.6
        filtered_group = group[group['score'] > 0.6]
        
        if len(filtered_group) > 0:
            # Trouver le texte de base avec la longueur la plus longue parmi les scores > 0.6
            base_text_idx = filtered_group['text'].apply(len).idxmax()
            base_row = filtered_group.loc[base_text_idx]
        else:
            # Si aucun score > 0.6, utiliser le texte de base avec la longueur la plus longue du groupe entier
            base_text_idx = group['text'].apply(len).idxmax()
            base_row = group.loc[base_text_idx]
        
        base_bbox = extract_bbox_coordinates(base_row['bbox'])
        base_center = calculate_bbox_center(base_bbox)
        joined_text = base_row['text']
        
        for idx, row in group.iterrows():
            if idx == base_text_idx:
                continue
            bbox = extract_bbox_coordinates(row['bbox'])
            center = calculate_bbox_center(bbox)
            distance = calculate_distance(base_center, center)
            if distance < distance_threshold:
                if len(joined_text) < 12:
                    joined_text += '-' + row['text']
        joined_text = joined_text.replace('F1', 'FI').replace('FL', 'FI')
        final_results.append({'filename': filename, 'text': joined_text})

    # Vérifier si final_results est vide
    if final_results:
        final_df = pd.DataFrame(final_results)
    else:
        final_df = pd.DataFrame(columns=['filename', 'text'])

    def transfo(text):
        if len(text) == 10 and text[2] != '-' and text[6] != '-':
            if '-' not in text:
                return f'{text[0:2]}-{text[2:6]}-{text[6:]}'
            else:
                return text
        elif len(text) == 11 and text[2] != '-' and text[6] == '-':
            return f'{text[0:2]}-{text[2:]}' 
        elif len(text) == 11 and text[2] == '-' and text[7] != '-':
            return f'{text[0:7]}-{text[7:]}'    
        else:
            return text       

    if 'text' in final_df.columns:
        final_df['text'] = final_df['text'].apply(transfo)
    else:
        raise KeyError("'text' column is missing in final_df")

    # Fusionner avec les résultats attendus
    merged = pd.merge(expected, final_df, on='filename', how='left', suffixes=('_expected', '_detected'))

    # Comparer les textes avec une tolérance de 2 erreurs
    def is_close_enough(expected, detected, tolerance=2):
        if pd.isna(detected):
            return 'Invalide'
        elif len(detected) < 8:
            return 'Invalide'
        elif Levenshtein.distance(sorted(set(expected)), sorted(set(detected))) <= tolerance:
            return 'Valide'
        elif Levenshtein.distance(sorted(set(expected)), sorted(set(detected))) <= tolerance + 2:
            return 'à Vérif'
        else:
            return 'Invalide'
        
    # Ajouter 'pas de PTO' pour les fichiers manquants dans 'detected'
    missing_files = expected[~expected['filename'].isin(detected['filename'])].copy()
    missing_files['correct'] = 'pas de PTO'
   
    # Supprimer les lignes 'merged' dont les filenames sont présents dans 'missing_files
    merged = merged[~merged['filename'].isin(missing_files['filename'])]
    merged['correct'] = merged.apply(lambda row: is_close_enough(row['text_expected'], row['text_detected']), axis=1)

    accuracy = (merged['correct'] == 'Valide').mean()

    df_fake = pd.DataFrame(fake, columns=['filename'])
    df_fake['correct'] = 'Montage'

    merged = pd.concat([missing_files, merged], ignore_index=True)
    final = pd.concat([df_fake, merged], ignore_index=True)

    # Afficher toutes les lignes
    pd.set_option('display.max_rows', None)

    print(f"Accuracy: {accuracy * 100:.2f}%")
    print(final[['filename', 'correct']])

    final.to_csv(os.path.join(output_dir, 'final_results.csv'), index=False)

    # Enregistrer la DataFrame dans un fichier Excel sans mise en forme
    excel_path = os.path.join(output_dir, 'final_results.xlsx')
    final[['filename', 'correct']].to_excel(excel_path, index=False)

    # Charger le fichier Excel pour appliquer la mise en forme conditionnelle
    wb = load_workbook(excel_path)
    ws = wb.active

    # Définir les styles de remplissage
    fill_green = PatternFill(start_color="00FF00", end_color="00FF00", fill_type="solid")
    fill_red = PatternFill(start_color="FF0000", end_color="FF0000", fill_type="solid")
    fill_orange = PatternFill(start_color="FFA500", end_color="FFA500", fill_type="solid")

    # Appliquer la mise en forme conditionnelle à la colonne 'correct'
    for row in ws.iter_rows(min_row=2, min_col=2, max_col=2):
        for cell in row:
            if cell.value == 'Valide':
                cell.fill = fill_green
            elif cell.value == 'à Vérif':
                cell.fill = fill_orange
            elif cell.value == 'Invalide' or cell.value == 'Montage' or cell.value == 'pas de PTO':
                cell.fill = fill_red

    # Sauvegarder le fichier Excel avec la mise en forme
    wb.save(excel_path)


#################################
# MAIN
#################################

# Fonction main
def main():
    # Dossier d'interventions à traiter 
    #input_directory = 'test_jeudi/Download_ARD2'
    # Dossier de sortie après traitement
    #output_directory = 'test_jeudi/outputs3'
   
    # Modèles de détection et de classification
    #model_yolo, model_spool, spool_class_names, device = load_models()
    
    # Process images
    #process_images(input_directory, output_directory, model_yolo, model_spool, spool_class_names, device)

    # Détection de fraudes sur PTO
    #fake = detect_fake_photomontages(output_directory)
   
    # Dossier pto_test pour aller chercher les photos valides de PTO
    #input_folder = os.path.join(output_directory, 'pto_test')

    # Dossier de retour pour les PTO avec numéro via OCR
    #output_folder = os.path.join(output_directory, 'results')

    # CSV des résultats OCR
    #results_file = os.path.join(output_directory, 'results_file.csv')
    #csv_with_numbers = os.path.join(output_directory, 'images_with_numbers.csv')

    # Comparaison des résultats
    #process_folder(input_folder, output_folder, results_file, csv_with_numbers, fake)
    #compare_ocr_results(output_directory, fake)
    
    # Dossier d'interventions à traiter 
    input_directory = 'test_jeudi/Download_ARD2'
    # Dossier de sortie après traitement
    output_directory = 'test_jeudi/outputs4'
   
    # Modèles de détection et de classification
    model_yolo, model_spool, spool_class_names, device = load_models()
    
    # Process images
    process_images(input_directory, output_directory, model_yolo, model_spool, spool_class_names, device)

    # Rechercher et déplacer les fichiers contenant "PTO" dans le dossier de destination
    pto_directory = os.path.join(output_directory, 'PTO')
    search_pto(output_directory, pto_directory)
    
    # Détection de fraudes sur PTO
    fake = detect_fake_photomontages(pto_directory)
   
    # Dossier pto_test pour aller chercher les photos valides de PTO
    input_folder = os.path.join(pto_directory, 'pto_test')

    # Dossier de retour pour les PTO avec numéro via OCR
    output_folder = os.path.join(pto_directory, 'results')

    # CSV des résultats OCR
    results_file = os.path.join(pto_directory, 'results_file.csv')
    csv_with_numbers = os.path.join(pto_directory, 'images_with_numbers.csv')

    # Comparaison des résultats
    process_folder(input_folder, output_folder, results_file, csv_with_numbers, fake)
    compare_ocr_results(pto_directory, fake)
    

if __name__ == "__main__":
    main()
